# Бенчмарк Cut Cross-Entropy

Аналог таблицы 1 и раздела 5.2 статьи: память и время последнего слоя (loss + градиент) для разных реализаций.

Порядок:
1. **Настройки и данные** — один раз прогоняет Gemma и сохраняет E, C, targets на диск; при повторном запуске модель не грузится.
2. **Проверка корректности** — каждый метод сравнивается с Baseline по loss и градиентам.
3. **Бенчмарк** — память и время в режимах loss / grad / loss+grad.
4. **Разреженность softmax** — почему фильтрация градиента почти ничего не теряет.

In [ ]:
%load_ext autoreload
%autoreload 2

import gc
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn.functional as F
import transformers
import triton

DEVICE = "cuda"
RESULTS = Path("results")
RESULTS.mkdir(exist_ok=True)

env = {
    "torch": torch.__version__,
    "triton": triton.__version__,
    "transformers": transformers.__version__,
    "gpu": torch.cuda.get_device_name(0),
}
(RESULTS / "env.json").write_text(json.dumps(env, indent=1))
print(env)
free, total = torch.cuda.mem_get_info()
print(f"Свободно памяти GPU: {free / 2**30:.1f} из {total / 2**30:.1f} ГБ")

## 0. Настройки

In [ ]:
MODEL_NAME = "google/gemma-2-2b-it"  # как в статье: Gemma 2 (2B) Instruct
DATA_CSV = "data/alpaca.csv"
MAX_LENGTH = 1024                    # обрезка примеров при токенизации
N_TOKENS = 8192                      # размер батча в токенах, как в таблице 1
SEED = 0
INPUTS_PATH = RESULTS / f"last_layer_inputs_N{N_TOKENS}.pt"

## 1. Данные → E, C, targets

E — выход бэкбона перед `lm_head` (только позиции с loss), C — веса `lm_head`, targets — правильные следующие токены.
Модель после извлечения удаляется, чтобы её веса не мешали замерам.

In [ ]:
from preprocessing import (
    LastLayerInputs,
    count_loss_tokens,
    extract_last_layer_inputs,
    load_examples,
    load_model_and_tokenizer,
    tokenize_examples,
)

if INPUTS_PATH.exists():
    inputs = LastLayerInputs.load(INPUTS_PATH, device=DEVICE)
    print("E, C, targets загружены из", INPUTS_PATH)
else:
    model, tokenizer = load_model_and_tokenizer(MODEL_NAME, device=DEVICE)
    examples = load_examples(DATA_CSV)
    tokenized = tokenize_examples(tokenizer, examples, max_length=MAX_LENGTH, mask_prompt=True)
    print(f"Примеров: {len(tokenized)}, токенов с loss: {count_loss_tokens(tokenized)}")

    inputs = extract_last_layer_inputs(model, tokenized, n_tokens=N_TOKENS, device=DEVICE, shuffle_seed=SEED)
    inputs.save(INPUTS_PATH)

    del model
    gc.collect()
    torch.cuda.empty_cache()

print(f"E {tuple(inputs.E.shape)} {inputs.E.dtype} | C {tuple(inputs.C.shape)} {inputs.C.dtype} | "
      f"targets {tuple(inputs.targets.shape)} | softcap = {inputs.softcap}")

## 2. Проверка корректности

Каждый метод сравнивается с Baseline на одних и тех же E, C, targets.
Ожидаем: разница loss порядка 1e-3 и меньше, косинус градиентов ≈ 1.
Варианты с фильтрацией могут чуть сильнее отличаться по ∇C — это и есть цена фильтрации.

In [ ]:
from bench import benchmark_all, compare_gradients, free_memory, lower_bound_mb, to_paper_table
from methods import METHODS, baseline_loss_fn

rows = []
for name, fn in METHODS.items():
    if fn is baseline_loss_fn:
        continue
    rows.append({"method": name, **compare_gradients(baseline_loss_fn, fn, inputs)})
    free_memory()

correctness = pd.DataFrame(rows).set_index("method")
correctness.to_csv(RESULTS / "bench_correctness.csv")
correctness

## 3. Бенчмарк (аналог таблицы 1)

Память — пик сверх того, что было выделено до замеряемой фазы; время — среднее по повторам после прогрева.
Lower bound — размер самих ∇E + ∇C: меньше этого не может занять ни один метод в режимах с backward.

In [ ]:
bench_df = benchmark_all(METHODS, inputs, modes=("loss", "grad", "loss+grad"), n_warmup=2, n_repeats=5)
bench_df.to_csv(RESULTS / "bench_raw.csv", index=False)

print(f"Lower bound (∇E + ∇C): {lower_bound_mb(inputs):.0f} MB")
table = to_paper_table(bench_df).round(1)
table.to_csv(RESULTS / "bench_table.csv")
table

Сравнение с таблицей 1 статьи (режим loss+grad; A100 80 ГБ, Gemma 2 2B, 8192 токена).

In [ ]:
# (память МБ, время мс) из таблицы 1 статьи. CCE (fp32 accum) сопоставляем со строкой CCE-Kahan.
PAPER_LOSS_GRAD = {
    "Baseline": (28000, 208),
    "torch.compile": (16000, 143),
    "CCE": (1164, 145),
    "CCE (no grad filter)": (1162, 357),
    "CCE (fp32 accum)": (2326, 160),
    "CCE-Kahan-FullC": (2326, 313),
    "CCE-Kahan-FullE": (2326, 292),
}

ours = bench_df[bench_df["mode"] == "loss+grad"].set_index("method")[["peak_mem_mb", "time_ms"]]
paper = pd.DataFrame(PAPER_LOSS_GRAD, index=["paper_mem_mb", "paper_time_ms"]).T
comparison = ours.join(paper).round(1)
comparison.to_csv(RESULTS / "bench_vs_paper.csv")
comparison

In [ ]:
lg = bench_df[bench_df["mode"] == "loss+grad"].set_index("method")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
lg["peak_mem_mb"].plot.barh(ax=axes[0], color="tab:blue")
axes[0].axvline(lower_bound_mb(inputs), color="black", ls="--", label="Lower bound (∇E + ∇C)")
axes[0].set_xlabel("Пиковая память, МБ")
axes[0].legend()

lg["time_ms"].plot.barh(ax=axes[1], xerr=lg["time_ms_std"], color="tab:orange")
axes[1].set_xlabel("Время, мс")

for ax in axes:
    ax.invert_yaxis()
    ax.set_ylabel("")
fig.suptitle(f"Loss + градиент: N = {inputs.n_tokens}, |V| = {inputs.C.shape[0]}, D = {inputs.C.shape[1]}")
fig.tight_layout()
fig.savefig(RESULTS / "bench_loss_grad.png", dpi=150)

## 4. Разреженность softmax (аналог рис. 3 и раздела 5.2)

Логиты здесь материализуются намеренно, ради анализа, поэтому считаем кусками по `CHUNK` токенов.
- средняя вероятность k-го по величине токена (рис. 3);
- доля элементов S и G, которые не меньше порога ε = 2⁻¹²;
- доля непустых блоков G (блок 128 × 128, как в конфигурациях ядер пакета) без сортировки словаря и с сортировкой по среднему логиту.

In [ ]:
from typing import Optional

EPS = 2.0 ** -12
CHUNK = 1024              # логиты куска: 1024 × 256 000 в fp32 ≈ 1 ГБ
BLOCK_N, BLOCK_V = 128, 128


def capped_logits(e: torch.Tensor, C: torch.Tensor, softcap: Optional[float]) -> torch.Tensor:
    """Логиты куска токенов в fp32 с softcap — так же, как в baseline."""
    logits = (e @ C.T).float()
    if softcap:
        logits = softcap * torch.tanh(logits / softcap)
    return logits


@torch.no_grad()
def softmax_stats(inputs: LastLayerInputs, eps: float = EPS, chunk: int = CHUNK) -> dict:
    """Средняя вероятность по рангу, доли S ≥ eps и |G| ≥ eps, средний логит каждого слова."""
    E, C, targets = inputs.E, inputs.C, inputs.targets
    N, V = E.shape[0], C.shape[0]
    prob_by_rank = torch.zeros(V, device=E.device, dtype=torch.float64)
    mean_logit = torch.zeros(V, device=E.device, dtype=torch.float64)
    n_s, n_g = 0, 0
    for i in range(0, N, chunk):
        logits = capped_logits(E[i:i + chunk], C, inputs.softcap)
        mean_logit += logits.sum(0).double()
        S = torch.softmax(logits, dim=-1)
        prob_by_rank += S.sort(dim=-1, descending=True).values.sum(0).double()
        n_s += int((S >= eps).sum())
        S[torch.arange(S.shape[0], device=S.device), targets[i:i + chunk]] -= 1.0  # теперь это G = S − Y
        n_g += int((S.abs() >= eps).sum())
        del logits, S
    return {
        "prob_by_rank": (prob_by_rank / N).cpu(),
        "mean_logit": mean_logit / N,
        "frac_S_ge_eps": n_s / (N * V),
        "frac_G_ge_eps": n_g / (N * V),
    }


@torch.no_grad()
def nonempty_block_fraction(
    inputs: LastLayerInputs,
    vocab_order: Optional[torch.Tensor] = None,
    eps: float = EPS,
    chunk: int = CHUNK,
    bn: int = BLOCK_N,
    bv: int = BLOCK_V,
) -> float:
    """Доля блоков G размером bn × bv, где есть хотя бы один |G| ≥ eps (такие блоки не пропускаются).

    vocab_order — перестановка словаря (как сортировка по среднему логиту в CCE); None — исходный порядок.
    """
    E, C, targets = inputs.E, inputs.C, inputs.targets
    if vocab_order is not None:
        C = C[vocab_order]
        new_pos = torch.empty_like(vocab_order)
        new_pos[vocab_order] = torch.arange(len(vocab_order), device=vocab_order.device)
        targets = new_pos[targets]  # номера правильных слов в новом порядке
    V = C.shape[0]
    pad_v = (-V) % bv
    n_full = E.shape[0] - E.shape[0] % bn  # берём только целые блоки токенов
    nonempty, total = 0, 0
    for i in range(0, n_full, chunk):
        e = E[i:min(i + chunk, n_full)]
        G = torch.softmax(capped_logits(e, C, inputs.softcap), dim=-1)
        G[torch.arange(G.shape[0], device=G.device), targets[i:i + e.shape[0]]] -= 1.0
        big = F.pad(G.abs() >= eps, (0, pad_v))
        blocks = big.view(e.shape[0] // bn, bn, (V + pad_v) // bv, bv).any(dim=3).any(dim=1)
        nonempty += int(blocks.sum())
        total += blocks.numel()
        del G, big
    return nonempty / total


stats = softmax_stats(inputs)
vocab_order = torch.argsort(stats["mean_logit"], descending=True)
sparsity = {
    "frac_S_ge_eps": stats["frac_S_ge_eps"],
    "frac_G_ge_eps": stats["frac_G_ge_eps"],
    "nonempty_blocks_unsorted": nonempty_block_fraction(inputs),
    "nonempty_blocks_sorted": nonempty_block_fraction(inputs, vocab_order=vocab_order),
}
(RESULTS / "sparsity.json").write_text(json.dumps(sparsity, indent=1))
pd.Series(sparsity).to_frame("value")

In [ ]:
prob = stats["prob_by_rank"].numpy()
ranks = range(1, len(prob) + 1)
n_above = int((prob >= EPS).sum())  # сколько первых по рангу токенов в среднем не ниже порога

fig, ax = plt.subplots(figsize=(7, 4))
ax.loglog(ranks, prob, label="средняя вероятность")
ax.axhline(EPS, color="red", ls="--", label="ε = 2⁻¹² (порог bf16)")
ax.axvline(n_above, color="gray", ls=":", label=f"ранг {n_above}")
ax.set_xlabel("k-й по вероятности токен (лог. шкала)")
ax.set_ylabel("вероятность (лог. шкала)")
ax.set_title("Аналог рис. 3: вероятности быстро падают ниже порога")
ax.legend()
fig.tight_layout()
fig.savefig(RESULTS / "sparsity_prob_by_rank.png", dpi=150)

## 5. Своя реализация

Когда `my_cce` будет готова: добавить её в `METHODS` (methods.py) и заново прогнать разделы 2–3.